In [1]:
import os
import pandas as pd
from PIL import Image
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms

In [2]:
img_dir = "/kaggle/input/artworks-artist-classification-soi-1010-2025/train/train"

In [3]:
class ArtDataset(Dataset):
    def __init__(self, df, img_dir, transform=None):
        self.df = df
        self.img_dir = img_dir
        self.transform = transform
        self.label_to_idx = {artist: i for i, artist in enumerate(df["artist"].unique())}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_id = row["id"]
        artist = row["artist"]

        img_path = os.path.join(self.img_dir, img_id + ".jpg")

        if not os.path.exists(img_path):
            raise FileNotFoundError(img_path)

        img = Image.open(img_path).convert("RGB")

        if self.transform:
            img = self.transform(img)

        label = self.label_to_idx[artist]

        return img, label

In [4]:
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])

In [5]:
df = pd.read_csv("/kaggle/input/artworks-artist-classification-soi-1010-2025/train.csv")
train_dataset = ArtDataset(df, img_dir, transform)
train_loader = DataLoader(
    train_dataset,
    batch_size=64,          
    shuffle=True,
    num_workers=4,          
    pin_memory=True         
)

In [6]:
class BiggerCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        def conv_block(in_c, out_c):
            return nn.Sequential(
                nn.Conv2d(in_c, out_c, 3, padding=1),
                nn.BatchNorm2d(out_c),
                nn.ReLU(inplace=True),
                nn.MaxPool2d(2)
            )
        self.net = nn.Sequential(
            conv_block(3,32),  
            conv_block(32,64),
            conv_block(64,128),
            conv_block(128,256)
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),  
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(256, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    def forward(self,x):
        x = self.net(x)
        x = self.classifier(x)
        return x


In [7]:
device = "cuda" if torch.cuda.is_available() else "cpu"

num_classes = df["artist"].nunique()

model = BiggerCNN(num_classes).to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [8]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=64,         
    shuffle=True,
    num_workers=4,          
    pin_memory=True         
)

In [9]:
import torch
print(torch.cuda.is_available())  # should print True
print(torch.cuda.get_device_name(0))  # e.g., "Tesla T4"

True
Tesla T4


In [10]:
epochs = 100

for epoch in range(epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1}/{epochs} | Loss: {total_loss/len(train_loader):.4f}")


Epoch 1/100 | Loss: 3.6226
Epoch 2/100 | Loss: 3.2984
Epoch 3/100 | Loss: 3.1410
Epoch 4/100 | Loss: 3.0316
Epoch 5/100 | Loss: 2.9410
Epoch 6/100 | Loss: 2.8732
Epoch 7/100 | Loss: 2.8124
Epoch 8/100 | Loss: 2.7725
Epoch 9/100 | Loss: 2.7185
Epoch 10/100 | Loss: 2.6720
Epoch 11/100 | Loss: 2.6285
Epoch 12/100 | Loss: 2.6086
Epoch 13/100 | Loss: 2.5515
Epoch 14/100 | Loss: 2.5372
Epoch 15/100 | Loss: 2.4908
Epoch 16/100 | Loss: 2.4790
Epoch 17/100 | Loss: 2.4417
Epoch 18/100 | Loss: 2.4213
Epoch 19/100 | Loss: 2.3955
Epoch 20/100 | Loss: 2.3655
Epoch 21/100 | Loss: 2.3519
Epoch 22/100 | Loss: 2.3407
Epoch 23/100 | Loss: 2.3202
Epoch 24/100 | Loss: 2.2964
Epoch 25/100 | Loss: 2.2652
Epoch 26/100 | Loss: 2.2520
Epoch 27/100 | Loss: 2.2232
Epoch 28/100 | Loss: 2.2189
Epoch 29/100 | Loss: 2.1811
Epoch 30/100 | Loss: 2.1837
Epoch 31/100 | Loss: 2.1701
Epoch 32/100 | Loss: 2.1653
Epoch 33/100 | Loss: 2.1377
Epoch 34/100 | Loss: 2.1182
Epoch 35/100 | Loss: 2.1218
Epoch 36/100 | Loss: 2.1022
E

In [11]:
class TestArtDataset(Dataset):
    def __init__(self, img_dir, transform=None):
        self.img_dir = img_dir
        self.transform = transform
        self.files = os.listdir(img_dir)

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        img_name = self.files[idx]
        img_path = os.path.join(self.img_dir, img_name)
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)
        return img, img_name.replace(".jpg", "")


In [12]:
test_img_dir = "/kaggle/input/artworks-artist-classification-soi-1010-2025/test/test"

test_dataset = TestArtDataset(test_img_dir, transform=transform)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=4, pin_memory=True)

In [13]:
model.eval()
predictions = []

with torch.no_grad():
    for images, ids in test_loader:
        images = images.to(device)
        outputs = model(images)
        probs = torch.softmax(outputs, dim=1)
        pred_labels = torch.argmax(probs, dim=1).cpu().numpy()

        for img_id, label_idx in zip(ids, pred_labels):
            predictions.append((img_id, label_idx))

In [14]:
idx_to_label = {v: k for k, v in train_dataset.label_to_idx.items()} 

submission_data = []
for img_id, label_idx in predictions:
    artist_name = idx_to_label[label_idx]
    submission_data.append({"id": img_id, "artist": artist_name})

In [15]:
submission_df = pd.DataFrame(submission_data)
submission_df.to_csv("submission.csv", index=False)
print(submission_df.head())

    id             artist
0  iwl     Francisco Goya
1  czj          Rembrandt
2  arp       Diego Rivera
3  oqj  Amedeo Modigliani
4  lxr        Edgar Degas
